In [ ]:
import pandas as pd

In [ ]:
# ── CONFIG ─────────────────────────────────────────────────────────────────
# Set these to match config/paths.yaml
KINSHIP_TABLE = "<set to cfg.ukb.kinship_table in config/paths.yaml>"
SPARSE_GRM    = "<set to cfg.gwas.sparse_grm in config/paths.yaml>"
# EXTERNAL: text file listing the MRI-cohort sample IDs to include in the GRM
MRI_SAMPLE_LIST = "<EXTERNAL: MRI-cohort sample-inclusion list (FID/IID)>"


In [ ]:
kinship = KINSHIP_TABLE

In [ ]:
k = pd.read_table(kinship, sep=' ')[['ID1', 'ID2', 'Kinship']]

In [ ]:
k = k[k.Kinship > 0.5**4.5]

In [ ]:
assert len(set(zip(k.ID1, k.ID2))) == len(k)

In [ ]:
samples = pd.read_table(MRI_SAMPLE_LIST, sep=' ')

In [ ]:
ids = set(k.ID1).union(k.ID2).union(samples.ID_1)
ids = dict(zip(ids, range(len(ids))))

In [ ]:
with open(SPARSE_GRM + '.grm.id', 'w') as f:
    for iid in ids:
        f.write(f"{iid} {iid}\n")

In [ ]:
k["ID1"] = k.ID1.apply(lambda x: ids[x])
k["ID2"] = k.ID2.apply(lambda x: ids[x])

In [ ]:
kl = list(k.apply(lambda x: (int(x.ID2), int(x.ID1), x.Kinship) if x.ID1 < x.ID2 else (int(x.ID1), int(x.ID2), x.Kinship), axis=1))

In [ ]:
with open(SPARSE_GRM + '.grm.sp', 'w') as f:
    for idx1, idx2, Kinship in kl:
        f.write(f"{idx1} {idx2} {Kinship * 2}\n")

In [ ]:
with open(SPARSE_GRM + '.grm.sp', 'a') as f:
    for iid in ids:
        f.write(f"{ids[iid]} {ids[iid]} 1.0000\n")